In [96]:
import numpy as np
import tensorflow as tf
import csv
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from tensorflow.keras import layers
from scipy.optimize import minimize

In [110]:
data = []

with open('training_data_predicted_params.csv', newline='') as csvfile:
    reader = csv.reader(csvfile)
    next(reader)
    for row in reader:
        float_row = [float(item) for item in row[1:]]
        data.append(float_row)

data = np.array(data)
print(data[0,0:6],data[0,7])


[  0.9037924  313.         288.           0.58414083   0.45581203
   7.        ] 99.2248062015504


In [111]:
param_bounds = [
    (0, 1),      # IR
    (50, 500),   # NG
    (50, 500),   # PS
    (0, 1),      # PC
    (0, 1),      # PM
    (3, 15)      # NMP
]

def normalize_params(X, bounds):
    X_norm = np.empty_like(X)
    for i, (min_val, max_val) in enumerate(bounds):
        X_norm[:, i] = (X[:, i] - min_val) / (max_val - min_val)
    return X_norm

def denormalize_params(X_norm, bounds):
    X = np.empty_like(X_norm)
    for i, (min_val, max_val) in enumerate(bounds):
        X[:, i] = X_norm[:, i] * (max_val - min_val) + min_val
    return X

In [112]:

X = data[:, 0:6]  # Hyperparameters
y = data[:, 7:]  # Results

scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_scaled = scaler_X.fit_transform(X)
X_normalized = normalize_params(X, param_bounds)
print(X_normalized)
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1))

model = keras.Sequential([
    layers.Input(shape=(6,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)
])

model.compile(optimizer='adam', loss='mse')

[[0.9037924  0.58444444 0.52888889 0.58414083 0.45581203 0.33333333]
 [0.9037924  0.58444444 0.52888889 0.58414083 0.45581203 0.33333333]
 [0.9037924  0.58444444 0.52888889 0.58414083 0.45581203 0.33333333]
 [0.9037924  0.58444444 0.52888889 0.58414083 0.45581203 0.33333333]
 [0.9037924  0.58444444 0.52888889 0.58414083 0.45581203 0.33333333]
 [0.9037924  0.58444444 0.52888889 0.58414083 0.45581203 0.33333333]
 [0.9037924  0.58444444 0.52888889 0.58414083 0.45581203 0.33333333]
 [0.9037924  0.58444444 0.52888889 0.58414083 0.45581203 0.33333333]
 [0.9037924  0.58444444 0.52888889 0.58414083 0.45581203 0.33333333]]


In [117]:
model.fit(X_normalized, y_scaled, epochs=3000,validation_split=0.2, verbose=0)

In [101]:
# y_pred = scaler_y.inverse_transform(model.predict(X_scaled))
# # Optimize input to maximize output using gradient ascent
# hyper = tf.Variable([[0.1]*6], dtype=tf.float32)  # Initial guess
# optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)

# for step in range(200):
#     with tf.GradientTape() as tape:
#         prediction = model(hyper)
#         loss = -prediction  # Negative because we want to maximize

#     grads = tape.gradient(loss, [hyper])
#     optimizer.apply_gradients(zip(grads, [hyper]))

#     # Clamp values between 0 and 1 (if required)
#     hyper.assign(tf.clip_by_value(hyper, 0.0, 1.0))

# # Output the result
# print("Best hyperparameters found:", hyper.numpy())
# print("Predicted maximum result:", model(hyper).numpy()[0][0])

In [119]:
# def objective(params_original_scale):
#     params = np.array(params_original_scale).reshape(1, -1)
#     params_scaled = scaler_X.transform(params)
#     pred_scaled = model.predict(params_scaled, verbose=0)
#     return -pred_scaled[0, 0]  # Negative because we want to maximize

def objective(normalized_params):
    params = denormalize_params(np.array(normalized_params).reshape(1, -1), param_bounds)
    params_scaled = scaler_X.transform(params)
    pred_scaled = model.predict(params_scaled, verbose=0)
    return -pred_scaled[0, 0]  # maximize

# Initial guess: average of input parameters
initial_guess = normalize_params(np.mean(X, axis=0).reshape(1, -1), param_bounds)[0]
bounds = [(0, 1)] * len(param_bounds)

# Bounds: use your domain knowledge or assume inputs ∈ [0, 1]
bounds = [(0, 1)] * 6

result = minimize(objective, initial_guess, bounds=bounds, method='L-BFGS-B')
best_params_normalized = result.x
best_params = denormalize_params(best_params_normalized.reshape(1, -1), param_bounds)
best_params_scaled = scaler_X.transform(best_params.reshape(1, -1))
best_result_scaled = model.predict(best_params_scaled, verbose=0)
best_result = scaler_y.inverse_transform(best_result_scaled)



# Run the optimization
# result = minimize(objective, initial_guess, bounds=bounds, method='L-BFGS-B')


# best_params = result.x
# best_params_scaled = scaler_X.transform(best_params.reshape(1, -1))
# best_result_scaled = model.predict(best_params_scaled, verbose=0)
# best_result = scaler_y.inverse_transform(best_result_scaled)

print("✅ Best parameters found (original scale):", best_params)
print("🎯 Predicted result with these params:", best_result[0, 0])

✅ Best parameters found (original scale): [[  0.9037924   50.         500.           0.77040534   0.54894429
    3.        ]]
🎯 Predicted result with these params: 143.80302
